# Neural–DTB Replication of the 2D and 3D Games

This notebook clones the GitHub branch, displays the supplied references, verifies the code, and runs the two- and three-player game experiments.

## 1. Clone the GitHub branch

In [ ]:
import pathlib, shutil, subprocess

REPO_URL = 'https://github.com/sun-mengwei/dtb-colab-experiments.git'
BRANCH = 'codex/game-dynamics-dtb'  # use 'main' after merge
REPO_DIR = pathlib.Path('/content/dtb-colab-experiments')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    REPO_URL, str(REPO_DIR),
], check=True)
PROJECT_DIR = REPO_DIR / 'dtb_game_dynamics_unnormalized'
assert (PROJECT_DIR / 'replicate_thesis_figures.py').exists(), PROJECT_DIR

In [ ]:
%cd /content/dtb-colab-experiments/dtb_game_dynamics_unnormalized
!python -m pip install -q -r requirements.txt

## 2. Read the reference and target beside the code

In [ ]:
from IPython.display import Image, display, Markdown
display(Image('references/target_figures_4_2_4_3.png'))
display(Image('references/three_player_game_definition.png'))
display(Image('references/target_figures_4_5_4_6.png'))
display(Markdown(open('references/README.md').read()))

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Verify the mathematics and implementation

In [ ]:
!python -m pytest

## 4. Reproduce both target experiments

The fast preset generates six Neural–DTB panels and a direct SDE baseline for each initial distribution.

In [ ]:
!python replicate_thesis_figures.py --device auto --output-root outputs/colab_replication

## 5. Display Figure 4.2-style uniform results

In [ ]:
root = 'outputs/colab_replication'
display(Image(f'{root}/figure_4_2_uniform/dtb_snapshots.png'))
display(Image(f'{root}/figure_4_2_uniform/sde_baseline_snapshots.png'))
display(Image(f'{root}/figure_4_2_uniform/diagnostics.png'))

## 6. Display Figure 4.3-style Gaussian results

In [ ]:
display(Image(f'{root}/figure_4_3_gaussian/dtb_snapshots.png'))
display(Image(f'{root}/figure_4_3_gaussian/sde_baseline_snapshots.png'))
display(Image(f'{root}/figure_4_3_gaussian/diagnostics.png'))

## 7. Inspect the saved diagnostics

In [ ]:
import numpy as np
data = np.load(f'{root}/figure_4_3_gaussian/history.npz')
print('snapshot times:', data['snapshot_times'])
print('DTB final mean:', data['snapshot_particles'][-1].mean(axis=0))
print('SDE final mean:', data['sde_baseline_particles'][-1].mean(axis=0))
print('final residual:', data['projection_residuals'][-1])
print('final rank:', data['retained_ranks'][-1])
print('final alpha norm:', data['alpha_norms'][-1])

## 8. Compute the three-player game

The validated preset uses h=0.005, 200 steps, m=128, and svd_rtol=1e-4.

In [ ]:
!python replicate_three_player_game.py --device auto --output-dir outputs/colab_three_player

In [ ]:
display(Image('outputs/colab_three_player/dtb_snapshots.png'))
display(Image('outputs/colab_three_player/sde_baseline_snapshots.png'))
display(Image('outputs/colab_three_player/diagnostics.png'))

In [ ]:
data3 = np.load('outputs/colab_three_player/history.npz')
print('3D snapshot times:', data3['snapshot_times'])
print('3D DTB final mean:', data3['snapshot_particles'][-1].mean(axis=0))
print('3D SDE final mean:', data3['sde_baseline_particles'][-1].mean(axis=0))
print('3D final residual:', data3['projection_residuals'][-1])

## 9. Compare 512, 1,024, and 5,000 particles

Set the switch to True to run the controlled sample-count study. All three cases use m=128 and svd_rtol=1e-3; the 5,000-particle CPU run can take several minutes.

In [ ]:
RUN_SAMPLE_COUNT_STUDY = False
if RUN_SAMPLE_COUNT_STUDY:
    for n in (512, 1024, 5000):
        !python replicate_three_player_game.py --particles {n} --svd-rtol 1e-3 --skip-sde-baseline --device auto --output-dir outputs/sample_study_stable/n{n}
    !python compare_sample_counts.py
    display(Image('outputs/sample_study_stable/sample_count_comparison.png'))

## 10. Optional denser tangent basis

Enable this only after the fast run succeeds.

In [ ]:
RUN_PAPER_SCALE = False
if RUN_PAPER_SCALE:
    !python replicate_thesis_figures.py --paper-scale --device auto --output-root outputs/paper_scale
    !python replicate_three_player_game.py --paper-scale --device auto --output-dir outputs/paper_scale_three_player

## 11. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!zip -qr dtb_game_replication.zip outputs/colab_replication outputs/colab_three_player outputs/sample_study_stable
!cp dtb_game_replication.zip '/content/drive/MyDrive/'